# CS2309 — SwiftEdit WebUI (Gradio)

Setup + chạy **demo web** [`scripts/app_gradio.py`](../scripts/app_gradio.py) trên **Mac MPS** hoặc **Google Colab T4**.

### Hai tab ứng dụng

| Tab | Chức năng |
|-----|----------|
| **Chỉnh sửa bằng prompt** | Upload ảnh + source/edit prompt (semantic edit) |
| **Xóa vật thể (khoanh vùng)** | Cọ tô vùng cần xóa + prompt mô tả nền |

Tích hợp **fp16 + channels_last + EditCache** (SwiftEdit-RT).

### Mac (local)

1. Kernel **`.venv`** (Python 3.12), repo đã clone local
2. Cell **1** nhận `PROJECT_ROOT` → Cell **2** setup → Cell **3** mở WebUI tại `http://127.0.0.1:7860`

### Google Colab (T4)

1. **GPU T4** — Colab web: Runtime → T4; **extension:** New Colab Server → GPU → T4
2. Cell **1** clone repo → `/content/CS2309.CH201` (không chỉ upload `.ipynb` lẻ)
3. Cell **2** pip + weights — **tùy chọn Drive** (load nếu có / lưu sau khi tải)
4. Cell **3** launch Gradio với **`share=True`** → link `*.gradio.live` mở trên máy bạn

**Drive (Colab):** mặc định `USE_DRIVE=True` ở cell ②:
- Có sẵn `/content/drive/MyDrive/CS2309/swiftedit_weights` → symlink, **không** tải lại
- Chưa có → tải Qualcomm lần đầu → **lưu lên Drive** (`SAVE_WEIGHTS_TO_DRIVE=True`)

Repo private: Colab Secrets → `GITHUB_TOKEN`. Chỉnh `REPO_SLUG` ở cell 1 nếu fork.

### ① Clone / nhận repo + kiểm tra GPU (Colab)

In [ ]:
import os
import subprocess
import sys
from pathlib import Path


try:
    import ipywidgets as widgets
except ImportError:
    !pip install ipywidgets
    import ipywidgets as widgets
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


_COLAB_GPU_ERR = (
    "Colab chưa có GPU.\n"
    "• Colab web: Runtime → Change runtime type → T4 GPU\n"
    "• Colab extension: Select Kernel → Colab → New Colab Server "
    "→ Hardware accelerator: GPU → T4, rồi Restart kernel\n"
    "• Đang nối server CPU: Remove Server, tạo server GPU mới"
)


def _check_colab_gpu() -> None:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0 or not (r.stdout or "").strip():
        raise RuntimeError(_COLAB_GPU_ERR)
    names = [ln.strip() for ln in r.stdout.strip().splitlines() if ln.strip()]
    print("GPU OK (nvidia-smi):", ", ".join(names))


REPO_SLUG = "NguyenKz/CS2309.CH201"
USE_PRIVATE_REPO = True


def _colab_repo_url():
    public_url = f"https://github.com/{REPO_SLUG}.git"
    if not USE_PRIVATE_REPO:
        return public_url
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
        if not token:
            print("Không lấy GITHUB_TOKEN — dùng repo public hoặc thêm secret Colab.")
            raise ValueError("Không lấy GITHUB_TOKEN — dùng repo public hoặc thêm secret Colab.")
        return f"https://{token}@github.com/{REPO_SLUG}.git"
    except Exception as e:
        text_input = widgets.Text(value='Enter GITHUB_TOKEN', description='Text:')
        token = text_input.value
        if token:
            return f"https://{token}@github.com/{REPO_SLUG}.git"
        else:
            print(
                "Không lấy GITHUB_TOKEN — dùng repo public hoặc thêm secret Colab.\n"
                f"Chi tiết: {e}\nFallback: {public_url}"
            )
    raise e

REPO_URL = _colab_repo_url() if IN_COLAB else f"https://github.com/{REPO_SLUG}.git"
COLAB_REPO_DIR = Path("/content/CS2309.CH201")

if IN_COLAB:
    _check_colab_gpu()
    if not (COLAB_REPO_DIR / "SwiftEdit" / "infer.py").exists():
        print(f"Cloning github.com/{REPO_SLUG} → {COLAB_REPO_DIR} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(COLAB_REPO_DIR)],
            check=True,
        )
    PROJECT_ROOT = COLAB_REPO_DIR
    os.environ.setdefault("HF_HOME", "/content/huggingface")
    os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "900")
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent
    elif not (PROJECT_ROOT / "SwiftEdit" / "infer.py").exists():
        for p in [Path.cwd(), *Path.cwd().parents]:
            if (p / "SwiftEdit" / "infer.py").exists():
                PROJECT_ROOT = p
                break

APP_SCRIPT = PROJECT_ROOT / "scripts" / "app_gradio.py"
print("IN_COLAB:", IN_COLAB)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("app_gradio.py:", APP_SCRIPT.is_file())
print("Weights OK:", (PROJECT_ROOT / "SwiftEdit/swiftedit_weights/inverse_ckpt-120k").is_dir())
if IN_COLAB and not (PROJECT_ROOT / "SwiftEdit" / "infer.py").is_file():
    raise FileNotFoundError("Clone xong nhưng thiếu SwiftEdit/ — push đủ repo lên GitHub.")

### ② Setup — pip, weights (Drive tùy chọn), HF, Gradio

**Colab — biến ở đầu cell code:**

| Biến | Mặc định | Ý nghĩa |
|------|----------|---------|
| `USE_DRIVE` | `True` | Mount Drive; ưu tiên weights trên Drive |
| `SAVE_WEIGHTS_TO_DRIVE` | `True` | Sau khi tải Qualcomm → copy lên Drive (lần sau khỏi tải) |
| `DRIVE_WEIGHTS` | `.../CS2309/swiftedit_weights` | Thư mục weights FP32 trên Drive |

Mac local: bỏ qua Drive, chạy `setup_macos.sh`.

In [ ]:
# --- Colab Drive (tắt nếu không muốn dùng Drive) ---
USE_DRIVE = True
SAVE_WEIGHTS_TO_DRIVE = True  # lần đầu tải xong → lưu Drive
DRIVE_WEIGHTS = Path("/content/drive/MyDrive/CS2309/swiftedit_weights")

LOCAL_WEIGHTS = PROJECT_ROOT / "SwiftEdit" / "swiftedit_weights"


def _weights_ok(path: Path) -> bool:
    return (
        (path / "inverse_ckpt-120k").is_dir()
        and (path / "sbv2_0.5").is_dir()
        and (path / "ip_adapter_ckpt-90k" / "ip_adapter.bin").is_file()
    )


def _symlink_weights(link: Path, target: Path) -> None:
    """Gắn link → Drive. Local thiếu/hỏng thì xóa rồi symlink; local đã OK thì giữ nguyên."""
    import shutil

    link.parent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        if link.resolve() == target.resolve() and _weights_ok(link):
            print(f"OK đã gắn: {link} → {target}")
            return
        link.unlink()
    elif link.exists():
        if _weights_ok(link):
            # Đã có bản local đủ — không cần (và không được) đè bằng symlink
            print(f"Local weights đã OK — giữ {link} (bỏ qua symlink Drive).")
            return
        if link.is_dir():
            print(f"Local chưa đủ / hỏng — xóa rồi symlink Drive:\n  rm -rf {link}")
            shutil.rmtree(link)
        else:
            link.unlink()

    link.symlink_to(target, target_is_directory=True)
    print(f"symlink {link} → {target}")


def _save_weights_to_drive(local: Path, drive: Path) -> None:
    import shutil

    if local.is_symlink():
        print("Local là symlink Drive — không copy lại.")
        return
    if not _weights_ok(local):
        print("Local weights chưa OK — bỏ qua lưu Drive.")
        return
    if _weights_ok(drive):
        print(f"Drive đã có weights: {drive}")
        return
    drive.parent.mkdir(parents=True, exist_ok=True)
    if drive.exists() and not _weights_ok(drive):
        print(f"Drive path tồn tại nhưng thiếu file — không ghi đè: {drive}")
        return
    print(f"Đang copy weights lên Drive (~10GB, có thể lâu)...\n  {local} → {drive}")
    shutil.copytree(local, drive)
    print("Đã lưu Drive:", drive)


env = os.environ.copy()
env["REPO_SLUG"] = REPO_SLUG
env["COLAB_REPO_DIR"] = str(COLAB_REPO_DIR)
if IN_COLAB:
    env.setdefault("HF_HOME", "/content/huggingface")
    env.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "900")

    if USE_DRIVE:
        from google.colab import drive

        if not Path("/content/drive/MyDrive").is_dir():
            print("Mount Google Drive...")
            drive.mount("/content/drive")
        else:
            print("Drive đã mount.")

        if _weights_ok(LOCAL_WEIGHTS):
            env["SWIFTEDIT_SKIP_WEIGHTS_DOWNLOAD"] = "1"
            print(f"Local weights OK — SKIP tải Qualcomm.\n  {LOCAL_WEIGHTS}")
            if SAVE_WEIGHTS_TO_DRIVE and not _weights_ok(DRIVE_WEIGHTS):
                print("Drive chưa có → sẽ lưu sau setup.")
        elif _weights_ok(DRIVE_WEIGHTS):
            _symlink_weights(LOCAL_WEIGHTS, DRIVE_WEIGHTS)
            env["SWIFTEDIT_SKIP_WEIGHTS_DOWNLOAD"] = "1"
            print("Dùng weights từ Drive — SKIP tải Qualcomm.")
        else:
            print(f"Drive chưa có weights tại {DRIVE_WEIGHTS}")
            print("→ sẽ tải Qualcomm (setup), rồi lưu Drive nếu SAVE_WEIGHTS_TO_DRIVE=True")

if IN_COLAB:
    setup_sh = PROJECT_ROOT / "scripts" / "setup_colab.sh"
else:
    setup_sh = PROJECT_ROOT / "scripts" / "setup_macos.sh"

print("Chạy:", setup_sh)
subprocess.run(["bash", str(setup_sh)], check=True, env=env, cwd=PROJECT_ROOT)

if IN_COLAB and USE_DRIVE and SAVE_WEIGHTS_TO_DRIVE:
    _save_weights_to_drive(LOCAL_WEIGHTS, DRIVE_WEIGHTS)

# Gradio (launch qua subprocess ở cell ③ — tránh nest_asyncio + uvicorn trên Py 3.12)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "gradio>=5,<6", "huggingface-hub<1.0"],
    check=True,
)
import gradio as gr

print("gradio:", gr.__version__)
print("Weights OK:", _weights_ok(LOCAL_WEIGHTS), LOCAL_WEIGHTS)
print("Setup OK — chạy cell Launch WebUI.")


### ③ Launch WebUI

- **Local:** mở `http://127.0.0.1:7860` (hoặc đổi `PORT_START`)
- **Colab:** đợi **vài phút nạp model**, rồi mới có link **`*.gradio.live`** (dòng `Public URL` / `Running on public URL` ở **cuối** output)
- Cell chạy `python -u scripts/app_gradio.py` — log không buffer
- Dừng server: **Interrupt kernel** (■) hoặc Runtime → Restart

**Nếu `CUDA out of memory`:** GPU còn model từ lần chạy cũ. **Restart kernel** → chạy lại ①②③.

Đổi `DTYPE`: `"fp16"` (mặc định) hoặc `"fp32"`.

In [ ]:
import socket

# --- Cấu hình ---
DTYPE = "fp16"  # "fp32" để so baseline
PORT_START = 7860


def _free_port(start: int = 7860, count: int = 20) -> int:
    for port in range(start, start + count):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                s.bind(("0.0.0.0", port))
                return port
            except OSError:
                continue
    return start


def _colab_gpu_used_ratio() -> float | None:
    if not IN_COLAB:
        return None
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0 or not r.stdout.strip():
        return None
    used, total = (float(x.strip()) for x in r.stdout.strip().split(","))
    print(f"GPU VRAM: {used:.0f} / {total:.0f} MiB đang dùng")
    return used / total if total else None


ratio = _colab_gpu_used_ratio()
if ratio is not None and ratio > 0.85:
    raise RuntimeError(
        "GPU gần đầy — thường do đã chạy cell cũ `build_app()` trong kernel.\n"
        "→ Restart kernel (Runtime → Restart), rồi chạy lại cell ①②③.\n"
        "→ Đảm bảo cell ③ in 'Lệnh: ... app_gradio.py' (KHÔNG có build_app)."
    )

PORT = _free_port(PORT_START)
# -u: unbuffered stdout — Colab mới in dần log + URL gradio.live
cmd = [
    sys.executable,
    "-u",
    str(APP_SCRIPT),
    "--dtype",
    DTYPE,
    "--port",
    str(PORT),
]
if IN_COLAB:
    cmd.append("--share")

run_env = os.environ.copy()
run_env["PYTHONUNBUFFERED"] = "1"

print("Đang nạp model + mở WebUI — lần đầu có thể mất vài phút...")
print("Lệnh:", " ".join(cmd))
print("\n" + "=" * 60)
if IN_COLAB:
    print("Colab: đợi vài phút nạp model, rồi hiện link *.gradio.live")
    print("(URL in SAU khi load model xong — kéo xuống cuối output)")
else:
    print(f"Local: mở http://127.0.0.1:{PORT}")
print("Tab 1: Chỉnh sửa bằng prompt  |  Tab 2: Xóa vật thể (khoanh vùng)")
print("Dừng server: Interrupt kernel (■)")
print("=" * 60 + "\n")

subprocess.run(cmd, cwd=PROJECT_ROOT, check=True, env=run_env)
